# Airline Passenger Satisfaction - Machine Learning Pipeline
This notebook implements an end-to-end Machine Learning pipeline using `train.csv` dataset.

### Pipeline Steps:
1. **Data Loading & Exploration**
2. **Data Cleaning & Handling Missing Values** (Median Imputation)
3. **Feature Encoding** (One-Hot & Label Encoding for Categorical Columns)
4. **Feature Scaling** (StandardScaler)
5. **Train-Validation Split** (80% Train, 20% Validation)
6. **Model Implementation & Evaluation**:
   - Logistic Regression
   - Decision Tree Classifier
   - K-Nearest Neighbors Classifier
   - Naive Bayes Classifier (Gaussian)
   - Support Vector Machine
7. **Evaluation Metrics**:
   - Accuracy
   - AUC Score (ROC AUC)
   - Precision
   - Recall
   - F1 Score
   - Matthews Correlation Coefficient (MCC)
8. **Model Export & Download Option** (Saving trained `.joblib` models with download links)


In [2]:
# 1. Load Dataset
import pandas as pd
import numpy as np
import os
import joblib
from IPython.display import FileLink, display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# Load Dataset
data_path = './datasource/train.csv'
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
display(df.head())


Dataset Shape: (103904, 24)


,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [3]:
# 2. Data Cleaning & Feature Encoding
print('Missing values before imputation:\n', df.isnull().sum()[df.isnull().sum() > 0])

# Impute numerical missing values with median
if 'Arrival Delay in Minutes' in df.columns and df['Arrival Delay in Minutes'].isnull().sum() > 0:
    median_val = df['Arrival Delay in Minutes'].median()
    df['Arrival Delay in Minutes'].fillna(median_val, inplace=True)

print('\nMissing values after imputation:', df.isnull().sum().sum())

# Drop identifier column 'id' if present
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

# Target Encoding
le_target = LabelEncoder()
df['satisfaction'] = le_target.fit_transform(df['satisfaction'])

# Categorical Feature Encoding (Gender, Customer Type, Type of Travel, Class)
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print('\nProcessed Dataset Preview:')
display(df_encoded.head())


Missing values before imputation:
 Arrival Delay in Minutes    310
dtype: int64

Missing values after imputation: 310


C:\Users\admin\AppData\Local\Temp\ipykernel_12696\1206588457.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Arrival Delay in Minutes'].fillna(median_val, inplace=True)



Processed Dataset Preview:


,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction,Gender_Male,Customer Type_disloyal Customer,Type of Travel_Personal Travel,Class_Eco,Class_Eco Plus
0,13,460,3,4,3,1,5,3,5,5,...,5,5,25,18.0,0,True,False,True,False,True
1,25,235,3,2,3,3,1,3,1,1,...,4,1,1,6.0,0,True,True,False,False,False
2,26,1142,2,2,2,2,5,5,5,5,...,4,5,0,0.0,1,False,False,False,False,False
3,25,562,2,5,5,5,2,2,2,2,...,4,2,11,9.0,0,False,False,False,False,False
4,61,214,3,3,3,3,4,5,5,3,...,3,3,0,0.0,1,True,False,False,False,False


In [6]:
# 3. Feature Scaling & Train-Validation Split (80/20)
from sklearn.impute import SimpleImputer

# Separate features and target
X = df_encoded.drop(columns=['satisfaction'])
y = df_encoded['satisfaction']

# Train-Validation Split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Handle any remaining NaNs in X using SimpleImputer (strategy='median')
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Scale the imputed features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Validation set shape: {X_val_scaled.shape}")
print(f"Remaining NaNs in X_train: np.isnan(X_train_scaled).sum()")


Training set shape: (83123, 23)
Validation set shape: (20781, 23)
Remaining NaNs in X_train: np.isnan(X_train_scaled).sum()


In [7]:
# 4. Model Training, Evaluation, and Saving Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

metrics_results = []
os.makedirs('saved_models', exist_ok=True)

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_val_scaled)
    y_proba = model.predict_proba(X_val_scaled)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_proba)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    mcc = matthews_corrcoef(y_val, y_pred)
    
    metrics_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'AUC Score': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
        'MCC Score': round(mcc, 4)
    })
    
    # Save Model
    model_path = os.path.join('saved_models', f"{name.lower().replace(' ', '_')}.joblib")
    joblib.dump(model, model_path)

# Display Summary Results Table
results_df = pd.DataFrame(metrics_results)
print('=== Classification Model Performance Comparison ===')
display(results_df)


C:\Users\admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


=== Classification Model Performance Comparison ===


,Model,Accuracy,AUC Score,Precision,Recall,F1 Score,MCC Score
0,Logistic Regression,0.8764,0.9281,0.8690,0.8418,0.8551,0.7477
1,Decision Tree,0.9439,0.9435,0.9307,0.9406,0.9356,0.8859
2,K-Nearest Neighbors,0.9280,0.9691,0.9464,0.8838,0.9140,0.8536
3,Naive Bayes,0.8614,0.9217,0.8562,0.8174,0.8364,0.7169
4,Support Vector Machine,0.9539,0.9892,0.9546,0.9384,0.9464,0.9061


In [8]:
# 5. Download Options for Trained Models
print('=== Download Links for Trained Models ===')
saved_files = os.listdir('saved_models')
for file_name in saved_files:
    file_path = os.path.join('saved_models', file_name)
    print(f'Model File: {file_name}')
    display(FileLink(file_path))


=== Download Links for Trained Models ===
Model File: decision_tree.joblib


D:\MyData\MTech\ZG565 Machine Learning\Assignment-2\saved_models\decision_tree.joblib

Model File: k-nearest_neighbors.joblib


D:\MyData\MTech\ZG565 Machine Learning\Assignment-2\saved_models\k-nearest_neighbors.joblib

Model File: logistic_regression.joblib


D:\MyData\MTech\ZG565 Machine Learning\Assignment-2\saved_models\logistic_regression.joblib

Model File: naive_bayes.joblib


D:\MyData\MTech\ZG565 Machine Learning\Assignment-2\saved_models\naive_bayes.joblib

Model File: support_vector_machine.joblib


D:\MyData\MTech\ZG565 Machine Learning\Assignment-2\saved_models\support_vector_machine.joblib